# GraphKV v5：数值验证与公平 prompt 诊断

先运行 CPU 数值测试，再加载本地模型。此 notebook 未伪造任何 GPU 运行结果。

修复原诊断中的四个混杂因素：不同路径使用相同 token 分块和回答起始标记；每次 probe/generate 使用全新拼接缓存；KL 标签与方向一致；明确区分 query 前后物理长度。

本测试不把 sequential 和 GraphKV 的 logits 相等作为正确性标准：二者注意力结构本来不同。正确性基准是 `tests/test_graphkv.py` 中独立构造的完整图注意力掩码。


In [ ]:
from pathlib import Path
import sys, subprocess, importlib

ROOT = Path.cwd()  # 将本 notebook 放在 v5 模块旁运行
assert (ROOT / "graph_kv_cache.py").exists(), ROOT
sys.path.insert(0, str(ROOT))
for name in ["graph_kv_cache", "graph_kv_adapter", "kv_compat", "pcw", "pcw_parallel"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], check=True)


## 本地模型配置

使用明确路径，不依赖目录排序。建议先 `sdpa`，再单独重启内核以 `flash_attention_2` 重跑并比较保存结果。默认 prompt 使用原仓库 `utils.py` 的 Tulu 格式；其他模型不能直接套用这一模板。


In [ ]:
MODEL_PATH = Path("/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT")
GPU_ID = 0
BACKEND = "sdpa"  # 或 flash_attention_2，需要对应依赖
MAX_NEW_TOKENS = 32
RUN_PLAIN_PROMPT_CONTROL = True

import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
assert transformers.__version__ == "4.50.0"
assert MODEL_PATH.is_dir()
assert torch.cuda.is_available() and GPU_ID < torch.cuda.device_count()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True,
    torch_dtype=torch.bfloat16, device_map={"": GPU_ID}, attn_implementation=BACKEND).eval()
from graph_kv_adapter import (build_graphkv_from_ids, graphkv_next_logits,
    greedy_generate_graphkv, _tokenize_one, input_device, kl_divergence)
import graph_kv_cache, graph_kv_adapter
print(torch.__version__, transformers.__version__, model.hf_device_map)
print(graph_kv_cache.__file__, graph_kv_adapter.__file__)
print("generation EOS:", model.generation_config.eos_token_id, "tokenizer EOS:", tokenizer.eos_token_id)


In [ ]:
contexts = [
    "- Title: Geography of France.\nFrance is a country in Europe.\n",
    "- Title: Paris.\nParis is the capital city of France.\n",
    "- Title: Germany.\nBerlin is the capital city of Germany.\n",
    "- Title: Eiffel Tower.\nThe Eiffel Tower is located in Paris.\n",
]
question = "What is the capital city of France?"
prefix_body = ("You are an intelligent AI assistant. Please answer questions based on the user's instructions. "
    "Below are some reference documents that may help you in answering the user's question.\n\n")
query_body = ("Please write a high-quality answer for the given question using only the provided search "
    "documents (some of which might be irrelevant). \n Question: " + question + " \n")
prompts = {"tulu": ("<|user|>\n" + prefix_body, query_body + "<|assistant|>\n")}
if RUN_PLAIN_PROMPT_CONTROL:
    prompts["plain"] = (prefix_body, query_body)
    prompts["answer_cue"] = (prefix_body, query_body + "Answer:")


## 三种注意力结构，保持相同 query token IDs

`sequential`：拼接同一份分块 token IDs；`independent`：仅 source；`graph_full`：source 加一轮 target。后两者不应被当作与 sequential 等价。额外的 target 副本是 GraphKV 方法的一部分。

`probe` 与 `generate` 独立组装 cache，不能把已经做过 query forward 的 cache 再用于重复输入 query。


In [ ]:
def summarize_logits(logits):
    probs = logits.float().softmax(-1)
    values, tokens = probs.topk(10, dim=-1)
    return [{"id": int(i), "text": tokenizer.decode([int(i)], skip_special_tokens=False),
             "p": float(p)} for p, i in zip(values[0], tokens[0])]

@torch.no_grad()
def run_case(prefix, query):
    device = input_device(model)
    tok = lambda text: _tokenize_one(tokenizer, text, device)
    p, q = tok(prefix), tok(query)
    sources = {f"s{i}": tok(text) for i, text in enumerate(contexts)}
    targets = {f"t{i}": value for i, value in enumerate(sources.values())}
    seq_ids = torch.cat([p, *sources.values(), q], dim=-1)
    logits = {"sequential": model(seq_ids, attention_mask=torch.ones_like(seq_ids),
                                  use_cache=False).logits[:, -1].float()}
    output = model.generate(seq_ids, attention_mask=torch.ones_like(seq_ids), do_sample=False,
                            max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.eos_token_id)
    seq_tokens = output[0, seq_ids.shape[-1]:].tolist()
    result = {"sequential": {"physical_input_length": seq_ids.shape[-1], "tokens": seq_tokens,
        "text_raw": tokenizer.decode(seq_tokens, skip_special_tokens=False)}}
    for label, ts, es in [
        ("independent", {}, []),
        ("graph_full", targets, [(s,t) for s in sources for t in targets]),
    ]:
        g = build_graphkv_from_ids(model=model, source_inputs=sources, target_inputs=ts,
                                   edges=es, prefix_input_ids=p)
        physical_before = g.build_query_cache().get_seq_length()
        logits[label] = graphkv_next_logits(model=model, graph_cache=g, query_input_ids=q)
        # This second probe must reproduce the first. It never reuses query-mutated KV.
        repeat = graphkv_next_logits(model=model, graph_cache=g, query_input_ids=q)
        assert torch.equal(logits[label], repeat)
        generated = greedy_generate_graphkv(model=model, tokenizer=tokenizer, graph_cache=g,
            query_input_ids=q, max_new_tokens=MAX_NEW_TOKENS, skip_special_tokens=False, return_details=True)
        assert g.build_query_cache().get_seq_length() == physical_before
        result[label] = {"layout": g.summary(), "physical_before_query": physical_before,
            "physical_after_query": physical_before + q.shape[-1], "query_tokens": q.tolist(),
            "tokens": generated.token_ids, "text_raw": generated.text, "stopped_on_eos": generated.stopped_on_eos}
        if label == "graph_full":
            permuted = graphkv_next_logits(model=model, graph_cache=g, query_input_ids=q,
                source_ids=list(reversed(sources)), target_ids=list(reversed(targets)))
            result[label]["permutation_max_logit_error"] = float((permuted-logits[label]).abs().max())
            # bf16/GPU reduction order can cause small numerical differences; report rather than claim exact invariance.
    for label, values in logits.items():
        result[label]["top10"] = summarize_logits(values)
        if tokenizer.eos_token_id is not None:
            result[label]["tokenizer_eos_probability"] = float(values.softmax(-1)[0,tokenizer.eos_token_id])
    result["KL_A_to_B"] = {f"{a} || {b}": kl_divergence(logits[a],logits[b])
        for a in logits for b in logits if a != b}
    return result

import json
results = {name: run_case(prefix, query) for name,(prefix,query) in prompts.items()}
report = {"model_path":str(MODEL_PATH), "backend":BACKEND, "torch":torch.__version__,
          "transformers":transformers.__version__, "results":results}
Path(f"diagnostic_v5_{BACKEND}.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
for prompt_name, values in results.items():
    print("\nPROMPT:", prompt_name)
    for name in ["sequential", "independent", "graph_full"]:
        print(name, repr(values[name]["text_raw"]), values[name]["top10"][:3])
    print(values["KL_A_to_B"])


## 如何解释结果

- plain EOS、Tulu 正常：支持“原 prompt 缺少生成边界”的解释，但仍需多例验证。
- 三种结构均 EOS：先检查模型/模板/权重；不能判定 GraphKV 算错。
- CPU 图掩码参考测试失败：先修实现，停止解释质量结果。
- 同一 prompt 下不同结构输出不同：正常现象，不能仅凭 KL 推断谁正确。
- 要比较论文指标，需相同模型、数据、检索排序、截断规则及评估脚本，单个 Paris 问题不足以证明复现。

若要隔离“只改变 position”这个变量，应固定图、token IDs、掩码与缓存构建方式，再设计明确的位置策略；旧 notebook 的 official 与 paper 路径同时改变多个环节，不能视作单变量因果实验。
